# 1 Extracting Data

### 1.0 Imports

In [3]:
import pandas as pd
from fitparse import FitFile
from collections import Counter
from pathlib import Path
from datetime import datetime, timedelta

In [4]:
# ============================================================
# Parameters
# ============================================================
first_date =  "2026-06-03"

LOCAL_OFFSET = pd.Timedelta(hours=2)

GARMIN_EPOCH = datetime(1989, 12, 31)

In [5]:
# ============================================================
# Select date and inspect WELLNESS files
# ============================================================
data_dir = Path(f"../data/{first_date}")
wellness_files = sorted(data_dir.glob("*_WELLNESS.fit"))

print(f"Found {len(wellness_files)} WELLNESS files\n")

summary = []

for file in wellness_files:
    fitfile = FitFile(str(file))

    stress_count = 0

    for _ in fitfile.get_messages("stress_level"):
        stress_count += 1

    summary.append({
        "file": file.name,
        "stress_records": stress_count
    })

summary_df = pd.DataFrame(summary)

display(summary_df)

Found 7 WELLNESS files



,file,stress_records
0,444927859217_WELLNESS.fit,477
1,444930358648_WELLNESS.fit,5
2,444931351296_WELLNESS.fit,4
3,444984229414_WELLNESS.fit,219
4,445043193231_WELLNESS.fit,231
5,445097413482_WELLNESS.fit,229
6,445149205969_WELLNESS.fit,196


### 1.1 Stress

In [7]:
def extract_stress(
    wellness_files,
    curr_date,
    local_offset
):
    """
    Extract stress data from all WELLNESS files for one day.
    """

    all_rows = []

    for file in wellness_files:

        fitfile = FitFile(str(file))

        for msg in fitfile.get_messages("stress_level"):

            row = msg.get_values()

            all_rows.append({
                "source_file": file.name,
                "timestamp": row["stress_level_time"] + local_offset,
                "stress": row["stress_level_value"],
                "body_battery": row.get("body_battery")
            })

    stress_df = pd.DataFrame(all_rows)

    # Garmin missing value
    stress_df["stress"] = (
        stress_df["stress"]
        .replace(-1, pd.NA)
    )

    stress_df = (
        stress_df
        .sort_values("timestamp")
        .drop_duplicates(
            subset="timestamp",
            keep="last"
        )
        .reset_index(drop=True)
    )

    stress_df = stress_df[
        stress_df["timestamp"].dt.date
        == pd.to_datetime(curr_date).date()
    ].copy()

    return stress_df

In [8]:
stress_df = extract_stress(
    wellness_files,
    first_date,
    LOCAL_OFFSET
)

print("Rows:", len(stress_df))
print("Start:", stress_df["timestamp"].min())
print("End:", stress_df["timestamp"].max())

display(stress_df.head())
display(stress_df.tail())

Rows: 1360
Start: 2026-06-03 00:01:00
End: 2026-06-03 23:59:00


,source_file,timestamp,stress,body_battery
0,444927859217_WELLNESS.fit,2026-06-03 00:01:00,12,None
1,444927859217_WELLNESS.fit,2026-06-03 00:02:00,20,None
2,444927859217_WELLNESS.fit,2026-06-03 00:04:00,22,None
3,444927859217_WELLNESS.fit,2026-06-03 00:05:00,39,None
4,444927859217_WELLNESS.fit,2026-06-03 00:07:00,16,None


,source_file,timestamp,stress,body_battery
1355,445149205969_WELLNESS.fit,2026-06-03 23:55:00,17,None
1356,445149205969_WELLNESS.fit,2026-06-03 23:56:00,15,None
1357,445149205969_WELLNESS.fit,2026-06-03 23:57:00,13,None
1358,445149205969_WELLNESS.fit,2026-06-03 23:58:00,17,None
1359,445149205969_WELLNESS.fit,2026-06-03 23:59:00,24,None


### 1.2 Monitor Data

In [10]:
def extract_monitoring(
    wellness_files,
    curr_date
):
    """
    Extract monitoring data from all WELLNESS files for one day.
    Includes heart rate and activity-related monitoring fields.
    """

    rows = []

    target_date = pd.to_datetime(curr_date).date()

    for file in wellness_files:

        fitfile = FitFile(str(file))

        for msg in fitfile.get_messages("monitoring"):

            row = msg.get_values()

            if "timestamp_16" not in row:
                continue

            timestamp = (
                datetime.combine(
                    target_date,
                    datetime.min.time()
                )
                + timedelta(seconds=row["timestamp_16"])
            )

            rows.append({
                "source_file": file.name,
                "timestamp": timestamp,

                "heart_rate": row.get("heart_rate"),

                "activity_type": row.get("activity_type"),
                "intensity": row.get("intensity"),

                "cycles": row.get("cycles"),
                "active_calories": row.get("active_calories"),
                "active_time": row.get("active_time")
            })

    monitoring_df = (
        pd.DataFrame(rows)
        .sort_values("timestamp")
        .drop_duplicates(subset="timestamp", keep="last")
        .reset_index(drop=True)
    )

    return monitoring_df

In [11]:
monitoring_df = extract_monitoring(
    wellness_files,
    first_date
)

print("Rows:", len(monitoring_df))
print("Start:", monitoring_df["timestamp"].min())
print("End:", monitoring_df["timestamp"].max())

display(monitoring_df.head())
display(monitoring_df.tail())

Rows: 1304
Start: 2026-06-03 00:00:56
End: 2026-06-03 18:12:12


,source_file,timestamp,heart_rate,activity_type,intensity,cycles,active_calories,active_time
0,445097413482_WELLNESS.fit,2026-06-03 00:00:56,72.0,None,NaN,NaN,NaN,NaN
1,445097413482_WELLNESS.fit,2026-06-03 00:01:56,71.0,None,NaN,NaN,NaN,NaN
2,445097413482_WELLNESS.fit,2026-06-03 00:02:56,74.0,None,NaN,NaN,NaN,NaN
3,445097413482_WELLNESS.fit,2026-06-03 00:03:56,73.0,None,NaN,NaN,NaN,NaN
4,445097413482_WELLNESS.fit,2026-06-03 00:04:56,80.0,None,NaN,NaN,NaN,NaN


,source_file,timestamp,heart_rate,activity_type,intensity,cycles,active_calories,active_time
1299,445097413482_WELLNESS.fit,2026-06-03 18:07:12,68.0,None,NaN,NaN,NaN,NaN
1300,445097413482_WELLNESS.fit,2026-06-03 18:09:12,76.0,None,NaN,NaN,NaN,NaN
1301,445097413482_WELLNESS.fit,2026-06-03 18:10:12,73.0,None,NaN,NaN,NaN,NaN
1302,445097413482_WELLNESS.fit,2026-06-03 18:11:12,72.0,None,NaN,NaN,NaN,NaN
1303,445097413482_WELLNESS.fit,2026-06-03 18:12:12,70.0,None,NaN,NaN,NaN,NaN


In [12]:
monitoring_df.notna().sum().sort_values(ascending=False)

source_file        1304
timestamp          1304
heart_rate         1161
activity_type        86
intensity            86
cycles               86
active_calories      86
active_time          86
dtype: int64

### 1.3 Body Battery

In [14]:
def extract_body_battery(
    wellness_files,
    curr_date,
    local_offset
):
    """
    Extract body battery data from all WELLNESS files for one day.
    """

    all_rows = []

    for file in wellness_files:

        fitfile = FitFile(str(file))

        for msg in fitfile.get_messages("stress_level"):

            row = msg.get_values()

            all_rows.append({
                "source_file": file.name,
                "timestamp": row["stress_level_time"] + local_offset,
                "body_battery": row["unknown_3"]
            })

    body_battery_df = pd.DataFrame(all_rows)

    body_battery_df["body_battery"] = (
        body_battery_df["body_battery"]
        .replace(-1, pd.NA)
    )

    body_battery_df = (
        body_battery_df
        .sort_values("timestamp")
        .drop_duplicates(
            subset="timestamp",
            keep="last"
        )
        .reset_index(drop=True)
    )

    body_battery_df = body_battery_df[
        body_battery_df["timestamp"].dt.date
        == pd.to_datetime(curr_date).date()
    ].copy()

    return body_battery_df

In [15]:
body_battery_df = extract_body_battery(
    wellness_files,
    first_date,
    LOCAL_OFFSET
)

print("Rows:", len(body_battery_df))
print("Start:", body_battery_df["timestamp"].min())
print("End:", body_battery_df["timestamp"].max())

display(body_battery_df.head())
display(body_battery_df.tail())

Rows: 1360
Start: 2026-06-03 00:01:00
End: 2026-06-03 23:59:00


,source_file,timestamp,body_battery
0,444927859217_WELLNESS.fit,2026-06-03 00:01:00,32
1,444927859217_WELLNESS.fit,2026-06-03 00:02:00,32
2,444927859217_WELLNESS.fit,2026-06-03 00:04:00,32
3,444927859217_WELLNESS.fit,2026-06-03 00:05:00,32
4,444927859217_WELLNESS.fit,2026-06-03 00:07:00,32


,source_file,timestamp,body_battery
1355,445149205969_WELLNESS.fit,2026-06-03 23:55:00,21
1356,445149205969_WELLNESS.fit,2026-06-03 23:56:00,21
1357,445149205969_WELLNESS.fit,2026-06-03 23:57:00,23
1358,445149205969_WELLNESS.fit,2026-06-03 23:58:00,23
1359,445149205969_WELLNESS.fit,2026-06-03 23:59:00,23


### Respirational Rate

In [17]:
def extract_respiration(
    wellness_files,
    curr_date,
    local_offset,
    garmin_epoch
):
    """
    Extract respiration data from all WELLNESS files for one day.
    """

    all_rows = []

    for file in wellness_files:

        fitfile = FitFile(str(file))

        for msg in fitfile.get_messages("unknown_297"):

            row = msg.get_values()

            all_rows.append({
                "source_file": file.name,
                "timestamp": (
                    garmin_epoch
                    + timedelta(seconds=row["unknown_253"])
                    + local_offset
                ),
                "respiration_rate": row["unknown_0"] / 100
            })

    respiration_df = pd.DataFrame(all_rows)

    respiration_df.loc[
        respiration_df["respiration_rate"] < 0,
        "respiration_rate"
    ] = pd.NA

    respiration_df = (
        respiration_df
        .sort_values("timestamp")
        .drop_duplicates(
            subset="timestamp",
            keep="last"
        )
        .reset_index(drop=True)
    )

    respiration_df = respiration_df[
        respiration_df["timestamp"].dt.date
        == pd.to_datetime(curr_date).date()
    ].copy()

    return respiration_df

In [18]:
respiration_df = extract_respiration(
    wellness_files,
    first_date,
    LOCAL_OFFSET,
    GARMIN_EPOCH
)

print("Rows:", len(respiration_df))
print("Start:", respiration_df["timestamp"].min())
print("End:", respiration_df["timestamp"].max())

display(respiration_df.head())
display(respiration_df.tail())

Rows: 1439
Start: 2026-06-03 00:01:00
End: 2026-06-03 23:59:00


,source_file,timestamp,respiration_rate
0,444927859217_WELLNESS.fit,2026-06-03 00:01:00,13.58
1,444927859217_WELLNESS.fit,2026-06-03 00:02:00,15.36
2,444927859217_WELLNESS.fit,2026-06-03 00:03:00,14.91
3,444927859217_WELLNESS.fit,2026-06-03 00:04:00,16.58
4,444927859217_WELLNESS.fit,2026-06-03 00:05:00,NaN


,source_file,timestamp,respiration_rate
1434,445149205969_WELLNESS.fit,2026-06-03 23:55:00,14.91
1435,445149205969_WELLNESS.fit,2026-06-03 23:56:00,15.25
1436,445149205969_WELLNESS.fit,2026-06-03 23:57:00,15.00
1437,445149205969_WELLNESS.fit,2026-06-03 23:58:00,14.33
1438,445149205969_WELLNESS.fit,2026-06-03 23:59:00,15.16


### 1.5 Merging

In [20]:
def build_daily_df(
    stress_df,
    body_battery_df,
    respiration_df,
    monitoring_df,
    curr_date
):
    """
    Merge all extracted Garmin data into a unified
    1-minute dataframe for one day.
    """

    # ========================================================
    # Ensure datetime dtype
    # ========================================================

    for df in [
        stress_df,
        body_battery_df,
        respiration_df,
        monitoring_df
    ]:
        df["timestamp"] = pd.to_datetime(df["timestamp"])

    # ========================================================
    # Keep only target day
    # ========================================================

    target_date = pd.to_datetime(curr_date).date()

    stress_df = stress_df[
        stress_df["timestamp"].dt.date == target_date
    ].copy()

    body_battery_df = body_battery_df[
        body_battery_df["timestamp"].dt.date == target_date
    ].copy()

    respiration_df = respiration_df[
        respiration_df["timestamp"].dt.date == target_date
    ].copy()

    monitoring_df = monitoring_df[
        monitoring_df["timestamp"].dt.date == target_date
    ].copy()

    # ========================================================
    # Full minute timeline
    # ========================================================

    full_index = pd.date_range(
        start=pd.Timestamp(curr_date),
        periods=1440,
        freq="1min"
    )

    # ========================================================
    # Helper
    # ========================================================

    def prepare_minute_df(df, value_col):

        temp = df.copy()

        temp["timestamp"] = temp["timestamp"].dt.floor("min")

        return (
            temp.groupby("timestamp")[value_col]
            .first()
            .to_frame()
        )

    # ========================================================
    # Build minute-level tables
    # ========================================================

    stress_min = prepare_minute_df(
        stress_df,
        "stress"
    )

    body_battery_min = prepare_minute_df(
        body_battery_df,
        "body_battery"
    )

    respiration_min = prepare_minute_df(
        respiration_df,
        "respiration_rate"
    )

    heart_rate_min = prepare_minute_df(
        monitoring_df,
        "heart_rate"
    )

    activity_type_min = prepare_minute_df(
        monitoring_df,
        "activity_type"
    )

    intensity_min = prepare_minute_df(
        monitoring_df,
        "intensity"
    )

    cycles_min = prepare_minute_df(
        monitoring_df,
        "cycles"
    )

    active_time_min = prepare_minute_df(
        monitoring_df,
        "active_time"
    )

    active_calories_min = prepare_minute_df(
        monitoring_df,
        "active_calories"
    )

    # ========================================================
    # Merge
    # ========================================================

    daily_df = pd.DataFrame(index=full_index)

    for feature_df in [
        stress_min,
        body_battery_min,
        respiration_min,
        heart_rate_min,
        activity_type_min,
        intensity_min,
        cycles_min,
        active_time_min,
        active_calories_min
    ]:
        daily_df = daily_df.join(feature_df)

    daily_df.index.name = "timestamp"

    return daily_df.reset_index()

In [21]:
daily_df = build_daily_df(
    stress_df,
    body_battery_df,
    respiration_df,
    monitoring_df,
    first_date
)

print("Shape:", daily_df.shape)

print("\nMissing values:")
print(daily_df.isna().sum())

print("\nDate range:")
print(daily_df["timestamp"].min())
print(daily_df["timestamp"].max())

display(daily_df.head(20))

Shape: (1440, 10)

Missing values:
timestamp              0
stress               221
body_battery          80
respiration_rate     307
heart_rate           498
activity_type       1355
intensity           1355
cycles              1355
active_time         1355
active_calories     1355
dtype: int64

Date range:
2026-06-03 00:00:00
2026-06-03 23:59:00


,timestamp,stress,body_battery,respiration_rate,heart_rate,activity_type,intensity,cycles,active_time,active_calories
0,2026-06-03 00:00:00,NaN,NaN,NaN,72.0,None,NaN,NaN,NaN,NaN
1,2026-06-03 00:01:00,12,32.0,13.58,71.0,None,NaN,NaN,NaN,NaN
2,2026-06-03 00:02:00,20,32.0,15.36,74.0,None,NaN,NaN,NaN,NaN
3,2026-06-03 00:03:00,NaN,NaN,14.91,73.0,None,NaN,NaN,NaN,NaN
4,2026-06-03 00:04:00,22,32.0,16.58,80.0,None,NaN,NaN,NaN,NaN
5,2026-06-03 00:05:00,39,32.0,NaN,76.0,None,NaN,NaN,NaN,NaN
6,2026-06-03 00:06:00,NaN,NaN,15.25,69.0,None,NaN,NaN,NaN,NaN
7,2026-06-03 00:07:00,16,32.0,13.75,71.0,None,NaN,NaN,NaN,NaN
8,2026-06-03 00:08:00,18,32.0,15.08,72.0,None,NaN,NaN,NaN,NaN
9,2026-06-03 00:09:00,19,32.0,14.75,NaN,walking,2.0,4645.0,5309.0,338.0


### 1.6 Save Function

In [23]:
def save_daily_df(
    daily_df,
    curr_date,
    output_dir="../final_dfs"
):
    """
    Save one extracted day as a CSV file.
    """

    output_dir = Path(output_dir)

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    output_file = output_dir / f"{curr_date}.csv"

    daily_df.to_csv(
        output_file,
        index=False
    )

    print(f"Saved: {output_file}")

### 1.7 Data Extraction

In [25]:
def extract_day(
    curr_date,
    local_offset,
    garmin_epoch
):
    """
    Complete extraction pipeline for one day.
    """

    data_dir = Path(f"../data/{curr_date}")

    wellness_files = sorted(
        data_dir.glob("*_WELLNESS.fit")
    )

    stress_df = extract_stress(
        wellness_files,
        curr_date,
        local_offset
    )

    body_battery_df = extract_body_battery(
        wellness_files,
        curr_date,
        local_offset
    )

    respiration_df = extract_respiration(
        wellness_files,
        curr_date,
        local_offset,
        garmin_epoch
    )

    monitoring_df = extract_monitoring(
        wellness_files,
        curr_date
    )

    daily_df = build_daily_df(
        stress_df,
        body_battery_df,
        respiration_df,
        monitoring_df,
        curr_date
    )

    return daily_df

### 1.8 Extracting multiple days at once

In [27]:
# ============================================================
# Process all dates
# ============================================================

dates = [
    "2026-06-03",
    "2026-06-04"
]

for curr_date in dates:

    print(f"\nProcessing {curr_date}")

    daily_df = extract_day(
        curr_date,
        LOCAL_OFFSET,
        GARMIN_EPOCH
    )

    save_daily_df(
        daily_df,
        curr_date
    )

    print("Shape:", daily_df.shape)

    print("\nMissing values:")
    print(daily_df.isna().sum())

    display(daily_df.head())


Processing 2026-06-03
Saved: ..\final_dfs\2026-06-03.csv
Shape: (1440, 10)

Missing values:
timestamp              0
stress               221
body_battery          80
respiration_rate     307
heart_rate           498
activity_type       1355
intensity           1355
cycles              1355
active_time         1355
active_calories     1355
dtype: int64


,timestamp,stress,body_battery,respiration_rate,heart_rate,activity_type,intensity,cycles,active_time,active_calories
0,2026-06-03 00:00:00,NaN,NaN,NaN,72.0,None,NaN,NaN,NaN,NaN
1,2026-06-03 00:01:00,12,32.0,13.58,71.0,None,NaN,NaN,NaN,NaN
2,2026-06-03 00:02:00,20,32.0,15.36,74.0,None,NaN,NaN,NaN,NaN
3,2026-06-03 00:03:00,NaN,NaN,14.91,73.0,None,NaN,NaN,NaN,NaN
4,2026-06-03 00:04:00,22,32.0,16.58,80.0,None,NaN,NaN,NaN,NaN



Processing 2026-06-04
Saved: ..\final_dfs\2026-06-04.csv
Shape: (1440, 10)

Missing values:
timestamp              0
stress               144
body_battery          54
respiration_rate     273
heart_rate           490
activity_type       1374
intensity           1374
cycles              1374
active_time         1374
active_calories     1374
dtype: int64


,timestamp,stress,body_battery,respiration_rate,heart_rate,activity_type,intensity,cycles,active_time,active_calories
0,2026-06-04 00:00:00,NaN,NaN,NaN,68.0,None,NaN,NaN,NaN,NaN
1,2026-06-04 00:01:00,NaN,NaN,15.00,77.0,None,NaN,NaN,NaN,NaN
2,2026-06-04 00:02:00,17,23.0,13.66,72.0,None,NaN,NaN,NaN,NaN
3,2026-06-04 00:03:00,20,23.0,14.16,70.0,None,NaN,NaN,NaN,NaN
4,2026-06-04 00:04:00,NaN,NaN,15.41,65.0,None,NaN,NaN,NaN,NaN


# 2 EDA

### Missing Values

In [30]:
daily_df.isna().sum()

timestamp              0
stress               144
body_battery          54
respiration_rate     273
heart_rate           490
activity_type       1374
intensity           1374
cycles              1374
active_time         1374
active_calories     1374
dtype: int64

# 3 Preprocessing

In [32]:
daily_df_processed = daily_df.copy()

### 3.1 Missing Values

In [34]:
# Fill activity features with forward fill
activity_cols = [
    "activity_type",
    "intensity",
    "cycles",
    "active_time",
    "active_calories"
]

daily_df_processed[activity_cols] = (
    daily_df_processed[activity_cols]
    .ffill()
)

In [35]:
daily_df_processed.isna().sum()

timestamp             0
stress              144
body_battery         54
respiration_rate    273
heart_rate          490
activity_type        10
intensity            10
cycles               10
active_time          10
active_calories      10
dtype: int64